# **Stroke Predictor App- Data Modelling process**

## Objectives

* Create Modelling Dataset
* Push data through ML pipeline for predictive analytics

## Inputs

* Dataset from the "Clean Data" file, imported from Kaggle

## Outputs

* Modelling Data, stored in "Modelling Data" file
* ML Model
    * Trained
    * Tested
    * Evaluated





---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\naqas\\OneDrive\\Documents\\Coding\\CI_Projects\\stroke_prediction_app\\stroke_prediction_app\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\naqas\\OneDrive\\Documents\\Coding\\CI_Projects\\stroke_prediction_app\\stroke_prediction_app'

---

# Import Dataset and Required Packages


In [13]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

In [5]:
clean_stroke_data_df = pd.read_csv("data_files/CleanData/stroke_data_clean.csv")
clean_stroke_data_df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,Male,81,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1
...,...,...,...,...,...,...,...,...,...,...,...
3420,Male,82,1,0,Yes,Self-employed,Rural,71.97,28.3,never smoked,0
3421,Female,57,0,0,Yes,Private,Rural,77.93,21.7,never smoked,0
3422,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
3423,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0


---

# _**Machine Learning Pipeline**_

## Encoding

As we will be building an ML model and analysing the impact of the variables on stroke likelihood then it is important to encode the categorical data  in new encoded coumns that are currently stored in the dataset as string values. This includes the following columns:

* Gender
* Ever Married
* Work Type
* Residence Type
* Smoking status

The first encoded columns to be created will be "Gender" ("Male" or "Female") "Ever Married" ("Yes" or "No") and "Residence Type" ("Urban" or "Rural") as they all have binary values.

In [6]:
clean_stroke_data_df["ever_married_encoded"] = clean_stroke_data_df["ever_married"].map({"Yes": 1, "No": 0})
clean_stroke_data_df["Residence_type_encoded"] = clean_stroke_data_df["Residence_type"].map({"Urban": 1, "Rural": 0})
clean_stroke_data_df["gender_encoded"] = clean_stroke_data_df["gender"].map({"Male": 1, "Female": 0})

clean_stroke_data_df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,ever_married_encoded,Residence_type_encoded,gender_encoded
0,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,1,1,1
1,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,1,0,1
2,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,1,1,0
3,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,1,0,0
4,Male,81,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3420,Male,82,1,0,Yes,Self-employed,Rural,71.97,28.3,never smoked,0,1,0,1
3421,Female,57,0,0,Yes,Private,Rural,77.93,21.7,never smoked,0,1,0,0
3422,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0,1,1,0
3423,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0,1,0,0


We can see in the new dataset that the new encoded columns have been added. Next, we will have to create encoded columns for the "Work Type" and "Smoking Status" columns. These will use one-hot encoding. Essentially this will create new columns for each variable within the original columns first. For example, the "Work Type" column will create five new encoded columns for the "Private", "Self-employed" ,"Govt", "Children" and "Never worked" variables.

In [7]:
work_type_encoded_temp_df = pd.get_dummies(clean_stroke_data_df["work_type"], prefix="work_type", dtype=int)
clean_stroke_data_df = pd.concat([clean_stroke_data_df, work_type_encoded_temp_df], axis=1)
clean_stroke_data_df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,ever_married_encoded,Residence_type_encoded,gender_encoded,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children
0,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,1,1,1,0,0,1,0,0
1,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,1,0,1,0,0,1,0,0
2,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,1,1,0,0,0,1,0,0
3,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,1,0,0,0,0,0,1,0
4,Male,81,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1,1,1,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3420,Male,82,1,0,Yes,Self-employed,Rural,71.97,28.3,never smoked,0,1,0,1,0,0,0,1,0
3421,Female,57,0,0,Yes,Private,Rural,77.93,21.7,never smoked,0,1,0,0,0,0,1,0,0
3422,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0,1,1,0,0,0,0,1,0
3423,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0,1,0,0,0,0,0,1,0


The columns have been added to the original dataframe by creating a temporary dataframe and concatenating it with the original dataframe. Now, to do the same with the "Smoking Status" column.

In [8]:
smoking_status_encoded_temp_df = pd.get_dummies(clean_stroke_data_df["smoking_status"], prefix="smoking_status",dtype= int)
clean_stroke_data_df = pd.concat([clean_stroke_data_df, smoking_status_encoded_temp_df], axis=1)
clean_stroke_data_df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,...,Residence_type_encoded,gender_encoded,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,...,1,1,0,0,1,0,0,1,0,0
1,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,...,0,1,0,0,1,0,0,0,1,0
2,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,...,1,0,0,0,1,0,0,0,0,1
3,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,...,0,0,0,0,0,1,0,0,1,0
4,Male,81,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,...,1,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3420,Male,82,1,0,Yes,Self-employed,Rural,71.97,28.3,never smoked,...,0,1,0,0,0,1,0,0,1,0
3421,Female,57,0,0,Yes,Private,Rural,77.93,21.7,never smoked,...,0,0,0,0,1,0,0,0,1,0
3422,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,...,1,0,0,0,0,1,0,0,1,0
3423,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,...,0,0,0,0,0,1,0,0,1,0


---

## Create new Dataframe from new dataset

Next, we will export the dataset as a new dataset and create a new dataframe from this dataset, primarly for modelling.

In [9]:
clean_stroke_data_df.to_csv("data_files/ModellingData/stroke_data_modelling.csv", index=False)

In [10]:
stroke_modelling_df = pd.read_csv("data_files/ModellingData/stroke_data_modelling.csv")
stroke_modelling_df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,...,Residence_type_encoded,gender_encoded,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,...,1,1,0,0,1,0,0,1,0,0
1,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,...,0,1,0,0,1,0,0,0,1,0
2,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,...,1,0,0,0,1,0,0,0,0,1
3,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,...,0,0,0,0,0,1,0,0,1,0
4,Male,81,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,...,1,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3420,Male,82,1,0,Yes,Self-employed,Rural,71.97,28.3,never smoked,...,0,1,0,0,0,1,0,0,1,0
3421,Female,57,0,0,Yes,Private,Rural,77.93,21.7,never smoked,...,0,0,0,0,1,0,0,0,1,0
3422,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,...,1,0,0,0,0,1,0,0,1,0
3423,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,...,0,0,0,0,0,1,0,0,1,0


It is important to drop the non-encoded columns as these will not be required for the modelling.

In [12]:
stroke_modelling_df = stroke_modelling_df.drop(["gender","ever_married","work_type","Residence_type","smoking_status"], axis=1)
stroke_modelling_df

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,ever_married_encoded,Residence_type_encoded,gender_encoded,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67,0,1,228.69,36.6,1,1,1,1,0,0,1,0,0,1,0,0
1,80,0,1,105.92,32.5,1,1,0,1,0,0,1,0,0,0,1,0
2,49,0,0,171.23,34.4,1,1,1,0,0,0,1,0,0,0,0,1
3,79,1,0,174.12,24.0,1,1,0,0,0,0,0,1,0,0,1,0
4,81,0,0,186.21,29.0,1,1,1,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3420,82,1,0,71.97,28.3,0,1,0,1,0,0,0,1,0,0,1,0
3421,57,0,0,77.93,21.7,0,1,0,0,0,0,1,0,0,0,1,0
3422,81,0,0,125.20,40.0,0,1,1,0,0,0,0,1,0,0,1,0
3423,35,0,0,82.99,30.6,0,1,0,0,0,0,0,1,0,0,1,0


---

## Define the Features and Target 

From the new dataframe it is important that we define what the features are and what the target is.

In [14]:
X = stroke_modelling_df.drop(["stroke"], axis = 1)
y = stroke_modelling_df["stroke"]

The "X" represents the features and requires the dropping of the "stroke" column. "X" is capitalised as it represents that it is a matrix and this is considered ML convention. The "y" is the "stroke" column as this is the variable that we are predicting and so the target.

---

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Process Summary

* It was determined that encoding of any data that is currently stored as string would be important as it could aid in the ML pipeline
* Encoding is usually done before the modelling, as part of the ML pipeline. This is done as the ML model will struggle to compute string variables and requires them to be quantified in some manner so that it can be understood by the model and used for the predictive analytics.
* As part of the encoding process the following columns had new encoded columns created:

    
    * Gender
    * Ever Married
    * Work Type
    * Residence Type
    * Smoking status
